# Build v1 — Full Label-Faithful (Lossless) TALIS 2024 Teacher Dataset

This notebook turns the raw `ttgintt4.csv` into **v1**: a faithful, fully labeled copy
(278,383 teachers x 630 variables), driven entirely by the codebook.

**What v1 does:** replace every numeric code with its codebook label, keeping **all** codes —
including `8 = Not administered`, `9 = Omitted or invalid`, `6 = Logically not applicable`,
`5 = I don't know`. Categorical vars become ordered `Categorical`; continuous/id/weight vars
are left exactly as the raw file. Original TALIS names are kept.

**What v1 does NOT do:** force anything to NaN, build the outcome, or apply modeling choices.
Deciding what counts as *missing* is done in **v2**.

*Source: OECD TALIS 2024 teacher public-use file. Cite: OECD (2025), TALIS 2024 Database, OECD, Paris.*

## 1. Imports & paths

In [1]:
from pathlib import Path
import re
import pandas as pd

# Absolute paths so the notebook runs no matter where it is saved.
DATA_DIR = Path("/Users/ruipinghuang/Desktop/9 Data Science/1 Data/TALIS2024_teachers_NoESE_CSV")
CODEBOOK = Path("/Users/ruipinghuang/Desktop/9 Data Science/5 Codebook/talis2024_teacher_codebook.csv")
OUT      = Path("v1_ttgintt4_labeled.parquet")          # output saved next to this notebook

data_path = DATA_DIR / "ttgintt4.csv"
print("raw data exists?", data_path.exists())
print("codebook exists?", CODEBOOK.exists())

raw data exists? True
codebook exists? True


## 2. Codebook + the label map

v1 is **lossless**: we keep every code and just attach its label. So we build the label
map from `all_value_labels`, which lists *all* codes (valid AND special/missing).

In [2]:
cb = pd.read_csv(CODEBOOK).set_index("variable_name")   # codebook is comma-separated

def value_labels(var):
    """Full {code: label} for TRUE categorical variables, covering ALL codes
       (valid AND special/missing). Returns {} for continuous / id / weight columns
       (they hold decimals and have no real answer labels) so they stay raw.

       Categorical-ness is decided by `valid_value_labels` (real answer codes); the labels
       come from `all_value_labels`, so special codes (8/9/6/5) are kept too."""
    if pd.isna(cb.loc[var, "valid_value_labels"]):
        return {}                                       # continuous / id / weight -> leave raw
    s = cb.loc[var, "all_value_labels"]                 # "1 = Yes | 2 = No | 8 = Not administered | ..."
    m = {}
    for part in str(s).split("|"):                      # split into "code = label" pieces
        mt = re.match(r"\s*(\d+)\s*=\s*(.*?)\s*$", part)   # group(1)=code, group(2)=label
        if mt:
            m[int(mt.group(1))] = mt.group(2)           # keep EVERY code, nothing dropped
    return dict(sorted(m.items()))                      # sorted by code -> categories in order

# ANNOTATION ONLY: which codes MEAN "missing" (so v2 knows what to NaN later).
# We do NOT apply this in v1 -- these codes are kept as their labels.
ADMIN = re.compile(r"not administered|omitted|invalid", re.I)
def missing_codes(var):
    return {c for c, lab in value_labels(var).items() if ADMIN.search(lab)}

# example: TT4G37A keeps all codes; missing_codes just flags 8 & 9 for later
print("labels :", value_labels("TT4G37A"))
print("missing-type (for v2 only):", missing_codes("TT4G37A"))

labels : {1: 'Yes', 2: 'No', 6: 'Logically not applicable', 8: 'Not administered', 9: 'Omitted or invalid'}
missing-type (for v2 only): {8, 9}


## 3. Load the raw data (all 630 columns)

In [3]:
df = pd.read_csv(data_path, sep=";", low_memory=False)   # sep=";" : TALIS uses semicolons
print(df.shape)                                          # (278383, 630)

(278383, 630)


## 4. Label every categorical column (keep all codes)

One loop: categorical variables get **all** their codes mapped to labels (ordered
`Categorical`); continuous / id / weight columns are left exactly as in the raw file.

In [4]:
n_cat = n_num = 0
for col in df.columns:
    lm = value_labels(col)
    if lm:                                                  # categorical: map EVERY code -> label
        df[col] = df[col].astype("Int64").map(lm)          # incl 'Not administered', etc.
        df[col] = pd.Categorical(df[col], categories=list(lm.values()), ordered=True)
        n_cat += 1
    else:                                                   # continuous / id / weight: left RAW
        n_num += 1                                          # (no labels exist; untouched)

print(f"categorical (all codes labeled): {n_cat} | numeric/id (raw): {n_num}")

categorical (all codes labeled): 458 | numeric/id (raw): 172


## 5. Sanity checks — special codes are KEPT as labels, not NaN

In [5]:
print("TT4G36  categories:", list(df["TT4G36"].cat.categories))    # incl 'Not administered','Omitted or invalid'
print("TT4G37A categories:", list(df["TT4G37A"].cat.categories))   # incl 'Logically not applicable'
print("T4SELF (continuous, raw) dtype:", df["T4SELF"].dtype, "| max =", df["T4SELF"].max())
print("\nTT4G36 counts (note the special-code labels are present):")
print(df["TT4G36"].value_counts(dropna=False))

TT4G36  categories: ['Yes', 'No', 'Not administered', 'Omitted or invalid']
TT4G37A categories: ['Yes', 'No', 'Logically not applicable', 'Not administered', 'Omitted or invalid']
T4SELF (continuous, raw) dtype: float64 | max = 9999.0

TT4G36 counts (note the special-code labels are present):
TT4G36
Not administered      185489
No                     53071
Yes                    36762
Omitted or invalid      3061
Name: count, dtype: int64


## 6. Save v1 (Parquet preserves the Categorical dtypes & labels)

In [6]:
df.to_parquet(OUT, index=False)
print("saved:", OUT, "| size MB:", round(OUT.stat().st_size / 1e6, 1))

saved: v1_ttgintt4_labeled.parquet | size MB: 119.2
